In [ ]:
!pip install seaborn

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.metrics import roc_curve, auc, confusion_matrix

%matplotlib inline 

In [ ]:
DATAPATH = "data/mushrooms.csv"

data = pd.read_csv(DATAPATH)
data.head()

In [ ]:
# plot to see if dataset is balanced or not
x = data["class"]
ax = sns.countplot(x=x, data=data)

In [ ]:
# preprocessing for analysis 
def plot_data(hue, data):
    for i, col in enumerate(data.columns):
        plt.figure(i)
        ax = sns.countplot(x=data[col], hue=hue, data=data)

hue = data["class"]
data_to_plot = data.drop("class", axis=1)

plot_data(hue, data_to_plot)

In [ ]:
## still preprocessing

# # check for null values 
# for col in data.columns:
#     print(f"{col}: {data[col].isnull().sum()}")

print("Checking of null values...")
total_nulls = 0;

for col in data.columns:
    null_count = data[col].isnull().sum()
    print(f"{col}: {null_count} nulls")
    total_nulls += null_count

print(f"\n Total null values in dataset: {total_nulls}")
if total_nulls == 0:
    print("No null values found")

In [ ]:
# use LabelEncoder to transform class into 1s and 0s

le = LabelEncoder()
data["class"] = le.fit_transform(data["class"])

data.head()

In [ ]:
# encode the rest of the dataset 

encoded_data = pd.get_dummies(data).astype(int)
encoded_data.head()


In [ ]:
## Modeling ... preparing data for a machine learning model

# separating features(X) from target/labels(y)
y = data["class"].values.reshape(-1, 1)
X = encoded_data.drop(["class"], axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
## applying logistic regression...algorithm for classification( binary classification)

from sklearn.linear_model import LogisticRegression

# initialize the model
logistic_reg = LogisticRegression()

logistic_reg.fit(X_train, y_train.ravel())

## get probabiltiy
y_prob = logistic_reg.predict_proba(X_test)[:,1]
y_pred = np.where(y_prob > 0.5, 1, 0)

In [ ]:
# showing how many mushrooms where correctly classified
log_confusion_matrix = confusion_matrix(y_test, y_pred)
log_confusion_matrix

In [ ]:
# using another matrix to check (roc ... receiver operating characteristics)

false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(false_positive_rate, true_positive_rate)
roc_auc

In [ ]:
## function to plot the roc_curve (to evaluate binary classification model)
def plot_roc(roc_auc):
    plt.figure(figsize=(7, 7))
    plt.title("Receiver Operating Characteristics")
    plt.plot(false_positive_rate, true_positive_rate, color="red", label="AUC = %0.2f" % roc_auc)
    plt.legend(loc="lower right")
    plt.plot([0, 1], [0, 1], linestyle="--", color="blue")

    plt.axis("tight")
    plt.ylabel("True Positive Rate")
    plt.xlabel("False Positive Rate")

plot_roc(roc_auc)

In [ ]:
# linear discriminant analysis(LDA)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train.ravel())

## get probabiltiy
y_prob_lda = lda.predict_proba(X_test)[:,1]
y_pred_lda = np.where(y_prob_lda > 0.5, 1, 0)

In [ ]:
# build the confusion  and check perfect classification
lda_confusion_matrix = confusion_matrix(y_test, y_pred_lda)
lda_confusion_matrix

In [ ]:
# build and show roc_curve

false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob_lda)
roc_auc_lda = auc(false_positive_rate, true_positive_rate)
roc_auc_lda

In [ ]:
plot_roc(roc_auc_lda)

In [ ]:
## Quadratic Discriminant Analysis (QDA)

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train, y_train.ravel())

## get probability
y_prob_qda = qda.predict_proba(X_test)[:,1]
y_pred_qda = np.where(y_prob_qda > 0.5, 1, 0) # convert to binary predictions

In [ ]:
# build confusion matrix to evalute model performance and check if qda is perfect classifier
qda_confusion_matrix = confusion_matrix(y_test, y_pred_qda)
qda_confusion_matrix

In [ ]:
# build and show roc_curve
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_prob_qda)
roc_auc_qda = auc(false_positive_rate, true_positive_rate)
roc_auc_qda

In [ ]:
plot_roc(roc_auc_qda)